# OAB Pipeline — Benchmark de Modelos de Linguagem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1/blob/main/notebook.ipynb)

**Pipeline:**
1. Setup
2. Preparo dos dados
3. Download dos modelos
4. Respostas dos candidatos
5. Curadoria
6. Avaliação discursiva
7. Resultados
8. Resumo e push para o Git

## 1. Setup

In [ ]:
import os

# Clona o repositório (pula se já existir)
if not os.path.exists('/content/oab_pipeline'):
    !git clone https://github.com/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1 /content/oab_pipeline

%cd /content/oab_pipeline

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets pandas tqdm

In [ ]:
import random
import numpy as np
import torch
import pandas as pd
from google.colab import userdata
from huggingface_hub import whoami, snapshot_download

from config import (
    MODELOS_CANDIDATOS, MODELO_CURADOR, MODELO_JUIZ,
    QUESTOES_DISCURSIVAS_CSV, QUESTOES_OBJETIVAS_CSV,
    CURADORIA_DISCURSIVAS_CSV, CURADORIA_OBJETIVAS_CSV,
    RESPOSTAS_DISCURSIVAS_CSV, RESPOSTAS_OBJETIVAS_CSV,
    AVALIACAO_DISCURSIVAS_CSV, ACCURACY_MODELOS_CSV,
    BENCHMARK_DISCURSIVAS_CSV,
    DISC_SLICE_START, DISC_SLICE_END,
    OBJ_SLICE_START, OBJ_SLICE_END,
    REPO_DIR, GITHUB_REPO,
)

# Reprodutibilidade
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Autenticação Hugging Face
token = userdata.get('HF_TOKEN')
if not token:
    raise ValueError('HF_TOKEN não encontrado. Configure nas Secrets do Colab.')
os.environ['HF_TOKEN'] = token
print(whoami())

# GPU obrigatória
if not torch.cuda.is_available():
    raise RuntimeError('GPU não disponível. Ative em Ambiente de execução > Alterar tipo de hardware > T4.')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Preparo dos dados

In [ ]:
import json
from datasets import load_dataset
from data_utils import salvar_csv

# --- Discursivas ---
dataset = load_dataset('maritaca-ai/oab-bench', name='questions')
df_disc_raw = pd.DataFrame(dataset['train']).iloc[DISC_SLICE_START:DISC_SLICE_END].reset_index(drop=True)

def montar_texto_questao(row):
    enunciado = str(row['statement']).strip()
    itens = row.get('turns', [])
    if isinstance(itens, list):
        itens = [str(i).strip() for i in itens if str(i).strip()]
    else:
        itens = []
    if not itens:
        return enunciado
    itens_fmt = '\n\n'.join(f'{n}. {item}' for n, item in enumerate(itens, 1))
    return f'{enunciado}\n\n{itens_fmt}'

df_disc = pd.DataFrame({
    'question_id': df_disc_raw['question_id'],
    'dataset': 'J1',
    'tipo_questao': 'discursiva',
    'categoria_dataset': df_disc_raw['category'],
    'texto_questao': df_disc_raw.apply(montar_texto_questao, axis=1),
})
salvar_csv(df_disc, QUESTOES_DISCURSIVAS_CSV)
display(df_disc.head())

# --- Objetivas ---
dataset_obj = load_dataset('eduagarcia/oab_exams')
df_obj_raw = pd.DataFrame(dataset_obj['train']).iloc[OBJ_SLICE_START:OBJ_SLICE_END].reset_index(drop=True)

def normalizar_choices(choices):
    return json.dumps(dict(zip(choices.get('label', []), choices.get('text', []))), ensure_ascii=False)

df_obj = pd.DataFrame({
    'question_id': df_obj_raw['id'],
    'dataset': 'J2',
    'tipo_questao': 'objetiva',
    'numero_questao': df_obj_raw['question_number'],
    'categoria_dataset': df_obj_raw['question_type'],
    'texto_questao': df_obj_raw['question'],
    'alternativas_json': df_obj_raw['choices'].apply(normalizar_choices),
    'gabarito_oficial': df_obj_raw['answerKey'],
})
salvar_csv(df_obj, QUESTOES_OBJETIVAS_CSV)
display(df_obj.head())

## 3. Download dos modelos

In [ ]:
# O set() elimina duplicatas — MODELO_CURADOR e MODELO_JUIZ são o mesmo modelo,
# sem o set() o download seria feito duas vezes
print('Baixando modelos uma única vez...')
for model_name in set(list(MODELOS_CANDIDATOS.values()) + [MODELO_CURADOR, MODELO_JUIZ]):
    print(f'Baixando: {model_name}')
    snapshot_download(repo_id=model_name, cache_dir='/content/models')

## 4. Respostas dos candidatos

In [ ]:
from tqdm import tqdm
from data_utils import carregar_csv, salvar_csv
from model_utils import load_model, unload
from pipeline import gerar_resposta_discursiva, gerar_resposta_objetiva

questoes_disc = carregar_csv(QUESTOES_DISCURSIVAS_CSV)
questoes_obj  = carregar_csv(QUESTOES_OBJETIVAS_CSV)

respostas_discursivas = []
resultados_obj = []

for nome_modelo, path_modelo in MODELOS_CANDIDATOS.items():
    print(f'Processando candidato: {nome_modelo}')
    candidato_model, candidato_tok = load_model(path_modelo)

    print('  → Discursivas...')
    for _, row in tqdm(questoes_disc.iterrows(), total=len(questoes_disc)):
        respostas_discursivas.append(
            gerar_resposta_discursiva(candidato_model, candidato_tok, row.to_dict(), nome_modelo)
        )

    print('  → Objetivas...')
    for _, row in tqdm(questoes_obj.iterrows(), total=len(questoes_obj)):
        resultados_obj.append(
            gerar_resposta_objetiva(candidato_model, candidato_tok, row.to_dict(), nome_modelo)
        )

    unload(candidato_model, candidato_tok)

df_resp_disc = pd.DataFrame(respostas_discursivas)
df_obj_res   = pd.DataFrame(resultados_obj)

salvar_csv(df_resp_disc, RESPOSTAS_DISCURSIVAS_CSV)
salvar_csv(df_obj_res,   RESPOSTAS_OBJETIVAS_CSV)

display(df_resp_disc.head())
display(df_obj_res.head())

## 5. Curadoria

In [ ]:
from tqdm import tqdm
from data_utils import carregar_csv, salvar_csv
from model_utils import load_model, unload
from pipeline import gerar_curadoria

questoes_disc = carregar_csv(QUESTOES_DISCURSIVAS_CSV)
questoes_obj  = carregar_csv(QUESTOES_OBJETIVAS_CSV)

curadoria_discursivas = []
curadoria_objetivas   = []

print('Processando curadoria...')
curador_model, curador_tok = load_model(MODELO_CURADOR)

for _, row in tqdm(questoes_disc.iterrows(), total=len(questoes_disc)):
    curadoria_discursivas.append(gerar_curadoria(curador_model, curador_tok, row.to_dict(), 'discursiva'))

df_cur_disc = pd.DataFrame(curadoria_discursivas)
salvar_csv(df_cur_disc, CURADORIA_DISCURSIVAS_CSV)
display(df_cur_disc.head())

for _, row in tqdm(questoes_obj.iterrows(), total=len(questoes_obj)):
    curadoria_objetivas.append(gerar_curadoria(curador_model, curador_tok, row.to_dict(), 'objetiva'))

df_cur_obj = pd.DataFrame(curadoria_objetivas)
salvar_csv(df_cur_obj, CURADORIA_OBJETIVAS_CSV)
display(df_cur_obj.head())

unload(curador_model, curador_tok)

## 6. Avaliação discursiva

In [ ]:
from tqdm import tqdm
from data_utils import carregar_csv, salvar_csv
from model_utils import load_model, unload
from pipeline import gerar_avaliacao_discursiva

questoes_disc  = carregar_csv(QUESTOES_DISCURSIVAS_CSV)
respostas_disc = carregar_csv(RESPOSTAS_DISCURSIVAS_CSV)
curadoria_disc = carregar_csv(CURADORIA_DISCURSIVAS_CSV)

juiz_model, juiz_tok = load_model(MODELO_JUIZ)
avaliacoes_discursivas = []

for _, resposta_row in tqdm(respostas_disc.iterrows(), total=len(respostas_disc)):
    resposta_row  = resposta_row.to_dict()
    questao_row   = questoes_disc.loc[questoes_disc['question_id'] == resposta_row['question_id']].iloc[0].to_dict()
    curadoria_row = curadoria_disc.loc[curadoria_disc['question_id'] == resposta_row['question_id']].iloc[0].to_dict()
    avaliacoes_discursivas.append(
        gerar_avaliacao_discursiva(juiz_model, juiz_tok, questao_row, resposta_row, curadoria_row)
    )

unload(juiz_model, juiz_tok)

df_avaliacao_disc = pd.DataFrame(avaliacoes_discursivas)
salvar_csv(df_avaliacao_disc, AVALIACAO_DISCURSIVAS_CSV)
display(df_avaliacao_disc.head())

## 7. Resultados

In [ ]:
from data_utils import carregar_csv, salvar_csv, area_confiavel

# Accuracy — objetivas
# groupby agrupa por modelo, mean() sobre booleanos calcula a proporção de acertos
# (True=1, False=0), reset_index(name='accuracy') transforma o índice em coluna
df_obj_res = carregar_csv(RESPOSTAS_OBJETIVAS_CSV)
accuracy = df_obj_res.groupby('modelo')['correto'].mean().reset_index(name='accuracy')
accuracy.to_csv(ACCURACY_MODELOS_CSV, index=False)
display(accuracy)

# Benchmark — discursivas por área
df_avaliacao = carregar_csv(AVALIACAO_DISCURSIVAS_CSV)
df_curadoria = carregar_csv(CURADORIA_DISCURSIVAS_CSV)

df_merged = df_avaliacao.merge(
    df_curadoria[['question_id', 'area_especialidade_equipe', 'confianca']],
    on='question_id',
    how='left'
)

df_merged['area_benchmark'] = df_merged.apply(area_confiavel, axis=1)

df_benchmark = (
    df_merged
    .groupby(['area_benchmark', 'modelo_candidato'])['nota']
    .mean()
    .reset_index()
    .pivot(index='area_benchmark', columns='modelo_candidato', values='nota')
    .round(2)
)
df_benchmark.loc['Mean'] = df_benchmark.mean().round(2)
df_benchmark.index.name   = 'Área de Especialidade'
df_benchmark.columns.name = None

salvar_csv(df_benchmark.reset_index(), BENCHMARK_DISCURSIVAS_CSV)
display(df_benchmark)

## 8. Resumo e push para o Git

In [ ]:
from data_utils import carregar_csv, git_push

csvs = {
    'questoes_discursivas':  QUESTOES_DISCURSIVAS_CSV,
    'questoes_objetivas':    QUESTOES_OBJETIVAS_CSV,
    'curadoria_discursivas': CURADORIA_DISCURSIVAS_CSV,
    'curadoria_objetivas':   CURADORIA_OBJETIVAS_CSV,
    'respostas_discursivas': RESPOSTAS_DISCURSIVAS_CSV,
    'respostas_objetivas':   RESPOSTAS_OBJETIVAS_CSV,
    'avaliacao_discursivas': AVALIACAO_DISCURSIVAS_CSV,
    'benchmark_discursivas': BENCHMARK_DISCURSIVAS_CSV,
}

print('Resumo dos CSVs gerados:')
for nome, path in csvs.items():
    print(f'  {nome}: {len(carregar_csv(path))} linhas')

# Push — persiste os resultados no repositório Git
# Configure GITHUB_TOKEN nas Secrets do Colab (mesmo local do HF_TOKEN)
github_token = userdata.get('GITHUB_TOKEN')
if github_token:
    git_push(REPO_DIR, GITHUB_REPO, github_token)
else:
    print('GITHUB_TOKEN não encontrado nas Secrets. CSVs salvos apenas localmente.')

print('\nPipeline finalizado com sucesso.')